# Case Study 1: Hospital Readmission Prediction

**Task:** Use Logistic Regression with L2 regularization on patient records to predict 30-day readmission. Evaluate with ROC-AUC and discuss the clinical trade-off between false negatives and false positives.

**Dataset:** Upload the supplied `hospital_readmissions_30k.csv` file when prompted.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files
uploaded = files.upload()

df = pd.read_csv("hospital_readmissions_30k.csv")
df.head()


In [ ]:
# Remove patient ID
df = df.drop("patient_id", axis=1)

# Split blood pressure into systolic and diastolic values
bp = df["blood_pressure"].str.split("/", expand=True).astype(float)
df["systolic_bp"] = bp[0]
df["diastolic_bp"] = bp[1]
df = df.drop("blood_pressure", axis=1)

X = df.drop("readmitted_30_days", axis=1)
y = df["readmitted_30_days"].map({"No": 0, "Yes": 1})

print("Class distribution:")
print(y.value_counts())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report
)

cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = Pipeline([
    ("preprocess", preprocess),
    ("logistic", LogisticRegression(
        penalty="l2",
        C=1.0,
        max_iter=2000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", round(auc, 4))


In [ ]:
# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Hospital Readmission ROC Curve")
plt.legend()
plt.show()


In [ ]:
# Compare different probability thresholds.
# Lowering the threshold catches more true readmissions, but can increase false alarms.
for threshold in [0.30, 0.40, 0.50, 0.60]:
    pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    print(
        f"Threshold={threshold:.2f} | "
        f"TP={tp}, FP={fp}, FN={fn}, TN={tn} | "
        f"Sensitivity={sensitivity:.3f}, Specificity={specificity:.3f}"
    )


In [ ]:
# Illustrative clinical cost analysis.
# Change these values to match the clinical/business setting.
cost_false_negative = 5
cost_false_positive = 1

results = []
for threshold in [i / 100 for i in range(10, 91)]:
    pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    total_cost = cost_false_negative * fn + cost_false_positive * fp
    results.append((threshold, total_cost, fp, fn))

cost_df = pd.DataFrame(
    results, columns=["threshold", "cost", "false_positives", "false_negatives"]
)
best = cost_df.loc[cost_df["cost"].idxmin()]

print("Illustrative cost-optimal threshold:", round(best["threshold"], 2))
print("False-negative cost:", cost_false_negative)
print("False-positive cost:", cost_false_positive)
print("Minimum total cost:", int(best["cost"]))

plt.figure(figsize=(7, 4))
plt.plot(cost_df["threshold"], cost_df["cost"])
plt.xlabel("Decision Threshold")
plt.ylabel("Total Cost")
plt.title("Threshold vs Illustrative Clinical Cost")
plt.show()


## Interpretation

- ROC-AUC measures ranking ability independently of one fixed decision threshold.
- A **false negative** means a patient who will be readmitted is predicted as low risk; this can delay follow-up or discharge planning.
- A **false positive** means a patient is flagged as high risk when they are not readmitted; this can consume clinical resources unnecessarily.
- If missing a high-risk patient is considered more costly, a lower decision threshold can be justified, but it will generally create more false positives.
- The costs used in the notebook are **illustrative**, not medical recommendations.
